Report 5


Anh Do

020416-2317

anhd@kth.se

In [259]:
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import numpy as np
from gurobipy import GRB
import gurobipy as gp
from pyomo.opt import SolverFactory

WLS = { # Please dont steal my credentials
    "WLSACCESSID": "1e6bdedb-f27d-4b0c-8a0d-24a8c94a8cb5",
    "WLSSECRET": "47ebeda5-b872-4365-9981-ef8044c28db5",
    "LICENSEID": 2707350,
}

# Problem 1

### ECP

In [249]:
def build_ecp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    
    # Container for ECP cuts
    m.ecp_cuts = pyo.ConstraintList()

    return m

In [250]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as solver:
    solver.set_options(WLS)
    solver.options.update(grb_params)
    
    model = build_ecp_model()
    solver.set_instance(model)
    
    tol = 1e-4
    max_iter = 1000
    iteration = 0

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Violation':<12}")
    print("-" * 40)

    while iteration < max_iter:
        iteration += 1
        
        # 1. Solve Master
        solver.solve(model)
        
        # 2. Get current values
        x_val = {i: pyo.value(model.x[i]) for i in model.I}
        current_obj = pyo.value(model.obj)
        
        # 3. Check Constraint Violation: sum(x^2) - 3 <= 0
        sum_sq = sum(x_val[i]**2 for i in model.I)
        g_val = sum_sq - 3
        
        if g_val <= tol:
            print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f} (Converged!)")
            break
        
        print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f}")
        
        # 4. Add Cut: g(x^k) + grad * (x - x^k) <= 0
        # g(x^k) = g_val
        # grad = 2 * x_val
        lhs = g_val + sum(2 * x_val[i] * (model.x[i] - x_val[i]) for i in model.I)
        
        model.ecp_cuts.add(lhs <= 0)
        solver.add_constraint(model.ecp_cuts[len(model.ecp_cuts)])

    print("-" * 40)
    print("Final Solution:")
    for i in model.I:
        print(f"x[{i}] = {pyo.value(model.x[i]):.6f}")

Iter  | Obj Value    | Violation   
----------------------------------------
1     | -12.000000   | 27.000000   
2     | -10.000000   | 27.000000   
3     | -9.583333    | 24.395833   
4     | -8.750000    | 13.062500   
5     | -8.750000    | 23.062500   
6     | -8.483333    | 16.076445   
7     | -7.950000    | 9.279863    
8     | -7.471173    | 13.059594   
9     | -7.301533    | 6.996698    
10    | -6.666427    | 6.375741    
11    | -6.426148    | 10.203227   
12    | -6.413840    | 3.738519    
13    | -6.275229    | 4.524293    
14    | -6.132209    | 7.552628    
15    | -6.000000    | 7.950853    
16    | -5.835908    | 3.137654    
17    | -5.823421    | 10.197482   
18    | -5.778358    | 4.701756    
19    | -5.755645    | 2.972649    
20    | -5.728472    | 3.433632    
21    | -5.530254    | 2.720442    
22    | -5.389863    | 3.220922    
23    | -5.387168    | 3.613622    
24    | -5.337083    | 1.650179    
25    | -5.293258    | 1.706800    
26    | -5.195933    | 

### OA (Outer Approximation)

In [272]:
def build_master_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.mu = pyo.Var(domain=pyo.Reals)

    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    
    # Linear constraint
    m.obj_const = pyo.Constraint(expr=-sum(m.x[i] for i in m.I) <= m.mu)
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    
    # Linearized quadratic constraint
    m.lin_quad_const = pyo.ConstraintList()

    return m


def build_NLP_model(y_fixed: dict):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=pyo.Reals)
    for i, val in y_fixed.items():
        m.x[i].fix(val)

    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)  

    # Quad constraint
    m.quad_const = pyo.Constraint(expr=sum([m.x[i]*m.x[i] for i in m.I]) - 3 <= 0)

    return m

def build_feasibility_model(y_fixed: dict):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=pyo.Reals)
    for i, val in y_fixed.items():
        m.x[i].fix(val)

    m.mu = pyo.Var(domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)

    # Quad constraint
    m.quad_const = pyo.Constraint(expr=sum([m.x[i]*m.x[i] for i in m.I]) - 3 <= m.mu)

    return m

def add_tangent_cut(model, solver, x_val):
    sum_sq = sum(x_val[i]*x_val[i] for i in model.I)
    g_val = sum_sq - 3
    lhs = g_val + sum(2 * x_val[i] * (model.x[i] - x_val[i]) for i in model.I)
    model.lin_quad_const.add(lhs <= 0)
    solver.add_constraint(model.lin_quad_const[len(model.lin_quad_const)])

def fmt_sol(x_dict, keys=None, ndigits=2, int_keys=(5, 6, 7, 8), value_width=5):
    """Fixed-width formatting for dict {i: value} (nice table printing).

    - All entries use the same width so columns line up.
    - Indices in int_keys are printed as integers (no decimals).
    """
    if x_dict is None:
        return "None"
    if keys is None:
        keys = sorted(x_dict.keys())
    int_keys = set(int_keys)

    parts = []
    for k in keys:
        v = x_dict.get(k, None)
        if v is None:
            val_str = f"{'None':>{value_width}}"
        elif k in int_keys:
            try:
                val_str = f"{int(round(float(v))):>{value_width}d}"
            except (TypeError, ValueError):
                val_str = f"{str(v):>{value_width}}"
        else:
            try:
                val_str = f"{float(v):>{value_width}.{ndigits}f}"
            except (TypeError, ValueError):
                val_str = f"{str(v):>{value_width}}"

        parts.append(f"{k:>2}: {val_str}")

    return "  ".join(parts)
    

In [273]:
with pyo.SolverFactory('gurobi_persistent', manage_env=True) as master_solver:
    master_solver.set_options(WLS)
    master_solver.options.update(grb_params)
    
    master_model = build_master_model()
    master_solver.set_instance(master_model)

    mu_UB = float('inf')
    mu_LB = -float('inf')
    best_solution = None
    best_iter = None
    tol = 1e-4
    max_iter = 100
    guess = {5: 0, 6: 0, 7: 0, 8: 0}

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Solution':<120} | {'Note':<12}")
    print("-" * 140)

    for iteration in range(max_iter):
        # 1. NLP or Feasibility
        NLP_solver = pyo.SolverFactory('gurobi_direct')
        NLP_model = build_NLP_model(guess)
        result = NLP_solver.solve(NLP_model, tee=False, load_solutions=False)

        if result.solver.termination_condition == pyo.TerminationCondition.optimal:
            NLP_model.solutions.load_from(result)
            method, x_k = 'NLP', {i: pyo.value(NLP_model.x[i]) for i in NLP_model.I}
            sol_str = fmt_sol(x_k, keys=list(NLP_model.I))
            print(f"{iteration+1:<5} | {pyo.value(NLP_model.obj):<12.6f} | {sol_str:<120} | {method:<12}")

            add_tangent_cut(master_model, master_solver, x_k)

        else:
            feas_solver = pyo.SolverFactory('gurobi_direct')
            feas_model = build_feasibility_model(guess)
            feas_result = feas_solver.solve(feas_model, tee=False)
            method, x_k = 'Feasibility', {i: pyo.value(feas_model.x[i]) for i in feas_model.I}
        
            sol_str = fmt_sol(x_k, keys=list(feas_model.I))
            print(f"{iteration+1:<5} | {pyo.value(feas_model.obj):<12.6f} | {sol_str:<120} | {method:<12}")

            add_tangent_cut(master_model, master_solver, x_k)

        if method == 'NLP' and pyo.value(NLP_model.obj) < mu_UB:
            best_solution = x_k
            best_iter = iteration + 1
            mu_UB = pyo.value(NLP_model.obj)
            master_model.mu.setub(mu_UB-tol)
            master_solver.update_var(master_model.mu)

        
        # 2. Solve Master
        result = master_solver.solve(master_model, load_solutions=False)
        if result.solver.termination_condition != pyo.TerminationCondition.optimal:
            print("Master solver terminated no more outer to cuts can be added:")
            break
        
        master_model.solutions.load_from(result)
        x_k = {i: pyo.value(master_model.x[i]) for i in master_model.I}

        # Linear shittery small fix to remove the looped solution.
        if abs(sum(x_k.values()) - 3) >= tol: # and guess == {i: pyo.value(master_model.x[i]) for i in [5,6,7,8]}:
            add_tangent_cut(master_model, master_solver, x_k)
            
        guess = {i: pyo.value(master_model.x[i]) for i in [5,6,7,8]}
        sol_str = fmt_sol(x_k, keys=list(master_model.I))
        print(f"{iteration+1:<5} | {pyo.value(master_model.obj):<12.6f} | {sol_str:<120} | {'Master':<12}")

        mu_LB = pyo.value(master_model.obj)
        master_model.mu.setlb(mu_LB)
        master_solver.update_var(master_model.mu)

        if abs(mu_UB - mu_LB) <= tol:
            print("-" * 140)
            print("Converged!")
            break


        

Iter  | Obj Value    | Solution                                                                                                                 | Note        
--------------------------------------------------------------------------------------------------------------------------------------------
1     | -3.464101    |  1:  0.87   2:  0.87   3:  0.87   4:  0.87   5:     0   6:     0   7:     0   8:     0                                   | NLP         
1     | -12.000000   |  1: -2.00   2:  2.00   3: -2.00   4:  2.00   5:     3   6:     3   7:     3   8:     3                                   | Master      
2     | 41.000000    |  1: -2.00   2: -0.00   3: -2.00   4: -0.00   5:     3   6:     3   7:     3   8:     3                                   | Feasibility 
2     | -10.464408   |  1: -1.89   2:  2.00   3:  2.00   4:  1.36   5:     1   6:     3   7:     0   8:     3                                   | Master      
3     | 16.000000    |  1: -0.00   2: -0.00   3: -0.00   4: -0.0

In [265]:
NLP_model.pprint()

1 RangeSet Declarations
    I : Dimen=1, Size=8, Bounds=(1, 8)
        Key  : Finite : Members
        None :   True :   [1:8]

1 Var Declarations
    x : Size=8, Index=I
        Key : Lower : Value              : Upper : Fixed : Stale : Domain
          1 :    -2 : 0.4998503204232219 :     2 : False :  True :  Reals
          2 :    -2 : 0.5001495715171066 :     2 : False :  True :  Reals
          3 :    -2 : 0.4998503204232219 :     2 : False :  True :  Reals
          4 :    -2 : 0.5001495715171066 :     2 : False :  True :  Reals
          5 :     0 :                0.0 :     3 :  True :  True :  Reals
          6 :     0 :                1.0 :     3 :  True :  True :  Reals
          7 :     0 :                0.0 :     3 :  True :  True :  Reals
          8 :     0 :                1.0 :     3 :  True :  True :  Reals

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : - (x[1] + x[2]

In [266]:
best_solution

{1: 0.4998503204232219,
 2: 0.5001495715171066,
 3: 0.4998503204232219,
 4: 0.5001495715171066,
 5: 0.0,
 6: 1.0,
 7: 0.0,
 8: 1.0}

# Problem 2

### ECP

In [243]:
def build_ecp_model_2():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.z = pyo.Var(m.I, domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    m.linear_const2 = pyo.Constraint(expr=sum(m.z[i] for i in m.I) - 3 <= 0)
    
    # Container for ECP cuts
    m.ecp_cuts = pyo.ConstraintList()

    return m

In [244]:
# ECP Iterative Loop for Problem 2 (Pyomo implementation)

# 0. Setup
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}
# Using 'gurobi_direct' or 'gurobi_persistent' without WLS options 
solver = pyo.SolverFactory('gurobi_direct')
solver.options.update(grb_params)

model = build_ecp_model_2()

tol = 1e-5
max_iter = 100
iteration = 0

print(f"{'Iter':<5} | {'Obj Value':<12} | {'Max Viol':<12}")
print("-" * 40)

while iteration < max_iter:
    iteration += 1
    
    # 1. Solve Master
    results = solver.solve(model, tee=False)
    
    if results.solver.termination_condition != pyo.TerminationCondition.optimal:
        print(f"Solver terminated with condition: {results.solver.termination_condition}")
        break
    
    # 2. Get current values
    x_val = {i: pyo.value(model.x[i]) for i in model.I}
    z_val = {i: pyo.value(model.z[i]) for i in model.I}
    current_obj = pyo.value(model.obj)
    
    # 3. Check Constraint Violations: x[i]^2 - z[i] <= 0
    # We track the maximum violation to determine convergence
    max_viol = -float('inf')
    violated_indices = []
    
    for i in model.I:
        viol = x_val[i]**2 - z_val[i]
        if viol > max_viol:
            max_viol = viol
        if viol > tol:
            violated_indices.append(i)
    
    # Print status
    if max_viol <= tol:
        print(f"{iteration:<5} | {current_obj:<12.6f} | {max_viol:<12.6f} (Converged!)")
        break
    
    print(f"{iteration:<5} | {current_obj:<12.6f} | {max_viol:<12.6f}")
    
    # 4. Add Cuts for ALL violated constraints
    # Constraint: x[i]^2 - z[i] <= 0
    # Linearization at x_bar: 
    # (x_bar^2 - z_bar) + 2*x_bar*(x - x_bar) - 1*(z - z_bar) <= 0
    # => 2*x_bar*x - z - x_bar^2 <= 0
    # => z >= 2*x_bar*x - x_bar^2
    
    for i in violated_indices:
        xi = x_val[i]
        # lhs = 2*xi*x - z - xi^2
        lhs = 2 * xi * model.x[i] - model.z[i] - xi**2
        model.ecp_cuts.add(lhs <= 0)

print("-" * 40)
print("Final Solution:")
for i in model.I:
    resid = pyo.value(model.x[i])**2 - pyo.value(model.z[i])
    print(f"x[{i}] = {pyo.value(model.x[i]):.6f}  z[{i}] = {pyo.value(model.z[i]):.6f}  (Viol: {resid:.2e})")


Iter  | Obj Value    | Max Viol    
----------------------------------------
1     | -12.000000   | 9.000000    
2     | -7.000000    | 4.000000    
3     | -6.000000    | 9.000000    
4     | -5.500000    | 1.000000    
5     | -5.416667    | 1.000000    
6     | -5.000000    | 1.000000    
7     | -4.000000    | 0.062500    
8     | -4.000000    | 0.015625    
9     | -4.000000    | 0.003906    
10    | -4.000000    | 0.000977    
11    | -4.000000    | 0.000244    
12    | -4.000000    | 0.000061    
13    | -4.000000    | 0.000015    
14    | -4.000000    | 0.000004     (Converged!)
----------------------------------------
Final Solution:
x[1] = 0.501953  z[1] = 0.251953  (Viol: 3.81e-06)
x[2] = 0.498047  z[2] = 0.248047  (Viol: 3.81e-06)
x[3] = 0.501953  z[3] = 0.251953  (Viol: 3.81e-06)
x[4] = 0.498047  z[4] = 0.248047  (Viol: 3.81e-06)
x[5] = -0.000000  z[5] = 0.000000  (Viol: 0.00e+00)
x[6] = 1.000000  z[6] = 1.000000  (Viol: 0.00e+00)
x[7] = -0.000000  z[7] = 0.000000  (Viol: 

### OA

In [267]:
def build_master_model_2():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.mu = pyo.Var(domain=pyo.Reals)
    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.z = pyo.Var(m.I, domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    m.linear_const2 = pyo.Constraint(expr=sum(m.z[i] for i in m.I) - 3 <= 0)
    m.obj_const = pyo.Constraint(expr=-sum(m.x[i] for i in m.I) <= m.mu)
    
    # Linearized quadratic constraint
    m.lin_quad_const = pyo.ConstraintList()

    return m


def build_NLP_model_2(y_fixed: dict):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=pyo.Reals)
    for i, val in y_fixed.items():
        m.x[i].fix(val)

    m.z = pyo.Var(m.I, domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraints
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    m.linear_const2 = pyo.Constraint(expr=sum(m.z[i] for i in m.I) - 3 <= 0)

    # Quadratic constraints (as nonlinear constraints in NLP)
    def quad_rule(m, i):
        return m.x[i] * m.x[i] <= m.z[i]
    m.quad_const = pyo.Constraint(m.I, rule=quad_rule)

    return m


def build_feasibility_model_2(y_fixed: dict):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    # Define x ONCE, then fix the integer components
    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=pyo.Reals)
    for i, val in y_fixed.items():
        m.x[i].fix(val)

    m.z = pyo.Var(m.I, domain=pyo.NonNegativeReals)
    m.mu = pyo.Var(domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    
    # Linear constraints
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    m.linear_const2 = pyo.Constraint(expr=sum(m.z[i] for i in m.I) - 3 <= 0)

    # Relaxed quadratic constraints with slack mu
    def quad_rule(m, i):
        return m.x[i] * m.x[i] - m.z[i] <= m.mu
    m.quad_const = pyo.Constraint(m.I, rule=quad_rule)

    return m


def add_tangent_cut_2(model, solver, x_val):
    # Iterate over every constraint i involving x[i]^2 <= z[i]
    for i in model.I:
        x_k = x_val[i]
        
        # Formulate the cut: 2*x_k*x - z - x_k^2 <= 0
        # Included derivation:
        # g(x,z) = x^2 - z <= 0
        # g(x_k, z_k) + grad_x * (x - x_k) + grad_z * (z - z_k) <= 0
        # (x_k^2 - z_k) + 2*x_k*(x - x_k) - 1*(z - z_k) <= 0
        # x_k^2 - z_k + 2*x_k*x - 2*x_k^2 - z + z_k <= 0
        # 2*x_k*x - z - x_k^2 <= 0

        lhs = 2 * x_k * model.x[i] - model.z[i] - x_k**2
        
        model.lin_quad_const.add(lhs <= 0)
        solver.add_constraint(model.lin_quad_const[len(model.lin_quad_const)])
    

In [271]:
with pyo.SolverFactory('gurobi_persistent', manage_env=True) as master_solver:
    master_solver.set_options(WLS)
    master_solver.options.update(grb_params)
    
    master_model = build_master_model_2()
    master_solver.set_instance(master_model)

    mu_UB = float('inf')
    mu_LB = -float('inf')
    best_solution = None
    best_iter = None
    tol = 1e-4
    max_iter = 150
    guess = {5: 0, 6: 0, 7: 0, 8: 0}

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Solution':<120} | {'Note':<12}")
    print("-" * 140)

    for iteration in range(max_iter):
        # 1. NLP or Feasibility
        NLP_solver = pyo.SolverFactory('gurobi_direct')
        NLP_model = build_NLP_model_2(guess)
        result = NLP_solver.solve(NLP_model, tee=False, load_solutions=False)

        if result.solver.termination_condition == pyo.TerminationCondition.optimal:
            NLP_model.solutions.load_from(result)
            method, x_k = 'NLP', {i: pyo.value(NLP_model.x[i]) for i in NLP_model.I}
            sol_str = fmt_sol(x_k, keys=list(NLP_model.I))
            print(f"{iteration+1:<5} | {pyo.value(NLP_model.obj):<12.6f} | {sol_str:<120} | {method:<12}")

            add_tangent_cut_2(master_model, master_solver, x_k)

        else:
            feas_solver = pyo.SolverFactory('gurobi_direct')
            feas_model = build_feasibility_model_2(guess)
            feas_result = feas_solver.solve(feas_model, tee=False)
            method, x_k = 'Feasibility', {i: pyo.value(feas_model.x[i]) for i in feas_model.I}
        
            sol_str = fmt_sol(x_k, keys=list(feas_model.I))
            print(f"{iteration+1:<5} | {pyo.value(feas_model.obj):<12.6f} | {sol_str:<120} | {method:<12}")

            add_tangent_cut_2(master_model, master_solver, x_k)

        if method == 'NLP' and pyo.value(NLP_model.obj) < mu_UB:
            best_solution = x_k
            best_iter = iteration + 1
            mu_UB = pyo.value(NLP_model.obj)
            master_model.mu.setub(mu_UB-tol)
            master_solver.update_var(master_model.mu)

        
        # 2. Solve Master
        result = master_solver.solve(master_model, load_solutions=False)
        if result.solver.termination_condition != pyo.TerminationCondition.optimal:
            print("Master solver terminated no more outer to cuts can be added:")
            break

        master_model.solutions.load_from(result)
        x_k = {i: pyo.value(master_model.x[i]) for i in master_model.I}

        # Linear shittery small fix to remove the looped solution ECP cut.
        if abs(sum(x_k.values()) - 3) >= tol: # and guess == {i: pyo.value(master_model.x[i]) for i in [5,6,7,8]}:
            add_tangent_cut_2(master_model, master_solver, x_k)
            
        guess = {i: pyo.value(master_model.x[i]) for i in [5,6,7,8]}
        sol_str = fmt_sol(x_k, keys=list(master_model.I))
        print(f"{iteration+1:<5} | {pyo.value(master_model.obj):<12.6f} | {sol_str:<120} | {'Master':<12}")

        mu_LB = pyo.value(master_model.obj)
        master_model.mu.setlb(mu_LB)
        master_solver.update_var(master_model.mu)

        if abs(mu_UB - mu_LB) <= tol:
            print("-" * 140)
            print("Converged!")
            break


        

Iter  | Obj Value    | Solution                                                                                                                 | Note        
--------------------------------------------------------------------------------------------------------------------------------------------
1     | -3.464099    |  1:  0.87   2:  0.87   3:  0.87   4:  0.87   5:     0   6:     0   7:     0   8:     0                                   | NLP         
1     | -10.597799   |  1: -2.00   2:  1.30   3: -2.00   4:  1.30   5:     3   6:     3   7:     3   8:     3                                   | Master      
2     | 8.250000     |  1: -2.00   2:  0.00   3: -2.00   4:  0.00   5:     3   6:     3   7:     3   8:     3                                   | Feasibility 
2     | -6.453723    |  1: -0.43   2:  1.23   3:  0.43   4:  1.23   5:     1   6:     1   7:     1   8:     1                                   | Master      
3     | 0.250000     |  1: -0.14   2:  0.00   3: -0.14   4:  0.0

In [269]:
best_solution

{1: 0.500045679620277,
 2: 0.49995426644060725,
 3: 0.5000456794380825,
 4: 0.49995426644060725,
 5: -0.0,
 6: 1.0,
 7: -0.0,
 8: 1.0}